# CMM Model Training

This notebook trains a Characteristic-Managed Momentum model following the paper:

- input `z_i,t`: monthly stock characteristics;
- input `r_i,t-252:t-22`: 231 daily log returns in the formation window;
- neural network output `z_hat_i,t`: one scalar per stock-month;
- weights `w_i,t-d = softmax(z_hat_i,t * r_i,t-d)`;
- signal `CMM_i,t = sum_d w_i,t-d * r_i,t-d)`;
- training target: next-month cross-sectional standardized return.

In [ ]:
from pathlib import Path
import json
import random

import numpy as np
import pandas as pd
import torch
from torch import nn

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('mps' if torch.backends.mps.is_available() else 'cuda' if torch.cuda.is_available() else 'cpu')
print('device:', DEVICE)

DATA_PATH = Path('../result/datasets/cmm_model_training_data.parquet')
FEATURE_COLS_PATH = Path('../result/datasets/cmm_feature_columns.txt')
RETURN_COLS_PATH = Path('../result/datasets/cmm_return_window_columns.txt')
MODEL_DIR = Path('../result/models/cmm')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 180)

## 1. Load Training Data

In [ ]:
feature_cols = FEATURE_COLS_PATH.read_text(encoding='utf-8').splitlines()
return_cols = RETURN_COLS_PATH.read_text(encoding='utf-8').splitlines()

id_cols = ['stock_id', 'signal_date', 'trade_date', 'exit_date', 'target_1m_ret', 'target_1m_ret_cs_z']
use_cols = id_cols + return_cols + feature_cols

df = pd.read_parquet(DATA_PATH, columns=use_cols)
df['signal_date'] = pd.to_datetime(df['signal_date'])
df['trade_date'] = pd.to_datetime(df['trade_date'])
df['exit_date'] = pd.to_datetime(df['exit_date'])

print(df.shape)
print(df['signal_date'].min(), df['signal_date'].max())
df.head()

In [ ]:
X_ret = df[return_cols].to_numpy(dtype=np.float32)
X_z = df[feature_cols].to_numpy(dtype=np.float32)
y = df['target_1m_ret_cs_z'].to_numpy(dtype=np.float32)

months = np.array(sorted(df['signal_date'].unique()))
month_indices = {m: np.flatnonzero(df['signal_date'].values == m) for m in months}

print('X_ret:', X_ret.shape)
print('X_z:', X_z.shape)
print('months:', len(months))

## 2. Define the Paper-Style CMM Network

The network only maps characteristics into one scalar `z_hat`. The final prediction is still a weighted average of past daily returns.

In [ ]:
class CMMNet(nn.Module):
    def __init__(self, n_features: int, hidden=(128, 64, 32), dropout=0.05):
        super().__init__()
        layers = []
        in_dim = n_features
        for h in hidden:
            layers.append(nn.Linear(in_dim, h))
            layers.append(nn.GELU())
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.Dropout(dropout))
            in_dim = h
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, z_features, ret_window):
        z_hat = self.net(z_features)                    # [N, 1]
        scores = z_hat * ret_window                     # [N, 231]
        weights = torch.softmax(scores, dim=1)          # [N, 231]
        cmm = torch.sum(weights * ret_window, dim=1)    # [N]
        return cmm, z_hat.squeeze(-1), weights


def cs_zscore_tensor(x, eps=1e-8):
    return (x - x.mean()) / (x.std(unbiased=False) + eps)

## 3. Training Design

The training strategy follows an expanding-window setup: use all available historical months up to a cutoff, reserve the last 30% of that historical window for validation, train on the earlier months, and predict the next out-of-sample block. After each block, the window expands forward.


In [ ]:
INITIAL_TRAIN_MONTHS = 60
VAL_FRACTION = 0.30
TEST_BLOCK_MONTHS = 24
MIN_TEST_BLOCK_MONTHS = 6

N_EPOCHS = 80
PATIENCE = 12
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
HIDDEN = (128, 64, 32)
DROPOUT = 0.05


def expanding_splits(month_array, initial_train_months=60, val_fraction=0.30, test_block_months=24, min_test_block_months=6):
    month_array = list(month_array)
    start = initial_train_months
    fold = 1
    while start < len(month_array):
        train_val = month_array[:start]
        remaining = len(month_array) - start
        block_size = min(test_block_months, remaining)
        if block_size < min_test_block_months:
            break

        val_size = max(1, int(len(train_val) * val_fraction))
        train = train_val[:-val_size]
        val = train_val[-val_size:]
        test = month_array[start:start + block_size]
        yield {
            'fold': fold,
            'train_months': train,
            'val_months': val,
            'test_months': test,
            'train_start': pd.Timestamp(train[0]),
            'train_end': pd.Timestamp(train[-1]),
            'val_start': pd.Timestamp(val[0]),
            'val_end': pd.Timestamp(val[-1]),
            'test_start': pd.Timestamp(test[0]),
            'test_end': pd.Timestamp(test[-1]),
        }
        start += block_size
        fold += 1


folds = list(expanding_splits(
    months,
    initial_train_months=INITIAL_TRAIN_MONTHS,
    val_fraction=VAL_FRACTION,
    test_block_months=TEST_BLOCK_MONTHS,
    min_test_block_months=MIN_TEST_BLOCK_MONTHS,
))

fold_summary = pd.DataFrame([
    {
        'fold': f['fold'],
        'train_months': len(f['train_months']),
        'val_months': len(f['val_months']),
        'test_months': len(f['test_months']),
        'train_start': f['train_start'],
        'train_end': f['train_end'],
        'val_start': f['val_start'],
        'val_end': f['val_end'],
        'test_start': f['test_start'],
        'test_end': f['test_end'],
    }
    for f in folds
])
fold_summary


## 4. Training Helpers

Each optimization step uses one full month of stocks. This keeps the loss cross-sectional: model output is standardized inside that month, then matched to `target_1m_ret_cs_z`.


In [ ]:
def month_tensors(month):
    idx = month_indices[month]
    z = torch.from_numpy(X_z[idx]).to(DEVICE)
    r = torch.from_numpy(X_ret[idx]).to(DEVICE)
    target = torch.from_numpy(y[idx]).to(DEVICE)
    return z, r, target


def run_epoch(model, month_list, optimizer=None):
    is_train = optimizer is not None
    model.train(is_train)
    losses = []
    shuffled = list(month_list)
    if is_train:
        random.shuffle(shuffled)

    for month in shuffled:
        z, r, target = month_tensors(month)
        if is_train:
            optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(is_train):
            pred, _, _ = model(z, r)
            pred_cs = cs_zscore_tensor(pred)
            loss = nn.functional.mse_loss(pred_cs, target)
            if is_train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))


@torch.no_grad()
def predict_months(model, month_list, split='test', fold=None):
    model.eval()
    pieces = []
    for month in month_list:
        idx = month_indices[month]
        z, r, _ = month_tensors(month)
        pred, z_hat, weights = model(z, r)
        pred_cs = cs_zscore_tensor(pred)
        out = df.iloc[idx][['stock_id', 'signal_date', 'trade_date', 'exit_date', 'target_1m_ret', 'target_1m_ret_cs_z']].copy()
        out['cmm_signal'] = pred.detach().cpu().numpy()
        out['cmm_signal_cs_z'] = pred_cs.detach().cpu().numpy()
        out['z_hat'] = z_hat.detach().cpu().numpy()
        out['max_weight'] = weights.max(dim=1).values.detach().cpu().numpy()
        out['split'] = split
        if fold is not None:
            out['fold'] = fold
        pieces.append(out)
    return pd.concat(pieces, ignore_index=True)


def train_one_fold(fold_cfg):
    fold = fold_cfg['fold']
    print(f"\n===== Fold {fold} =====")
    print(
        f"train {fold_cfg['train_start'].date()} to {fold_cfg['train_end'].date()} | "
        f"val {fold_cfg['val_start'].date()} to {fold_cfg['val_end'].date()} | "
        f"test {fold_cfg['test_start'].date()} to {fold_cfg['test_end'].date()}"
    )

    model = CMMNet(n_features=len(feature_cols), hidden=HIDDEN, dropout=DROPOUT).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

    best_val = np.inf
    best_state = None
    bad_epochs = 0
    history = []

    for epoch in range(1, N_EPOCHS + 1):
        train_loss = run_epoch(model, fold_cfg['train_months'], optimizer)
        val_loss = run_epoch(model, fold_cfg['val_months'], optimizer=None)
        row = {
            'fold': fold,
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
        }
        history.append(row)
        print(f"fold {fold:02d} epoch {epoch:03d} | train {train_loss:.6f} | val {val_loss:.6f}")

        if val_loss < best_val - 1e-5:
            best_val = val_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f'fold {fold:02d} early stop')
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    fold_model_dir = MODEL_DIR / 'fold_models'
    fold_model_dir.mkdir(parents=True, exist_ok=True)
    model_path = fold_model_dir / f'cmm_model_fold_{fold:02d}.pt'
    torch.save({
        'state_dict': model.state_dict(),
        'feature_cols': feature_cols,
        'return_cols': return_cols,
        'fold': fold,
        'train_start': str(fold_cfg['train_start'].date()),
        'train_end': str(fold_cfg['train_end'].date()),
        'val_start': str(fold_cfg['val_start'].date()),
        'val_end': str(fold_cfg['val_end'].date()),
        'test_start': str(fold_cfg['test_start'].date()),
        'test_end': str(fold_cfg['test_end'].date()),
        'best_val_loss': best_val,
    }, model_path)

    pred_test = predict_months(model, fold_cfg['test_months'], split='test', fold=fold)
    pred_val = predict_months(model, fold_cfg['val_months'], split='val', fold=fold)
    return model, pd.DataFrame(history), pred_test, pred_val


## 5. Run Training

This trains one model per fold and concatenates all out-of-sample test-block predictions. The saved `cmm_predictions.parquet` is the paper-style out-of-sample signal file used by downstream comparison notebooks.


In [ ]:
all_history = []
all_test_predictions = []
all_val_predictions = []
last_model = None

for fold_cfg in folds:
    model, history, pred_test, pred_val = train_one_fold(fold_cfg)
    last_model = model
    all_history.append(history)
    all_test_predictions.append(pred_test)
    all_val_predictions.append(pred_val)

history_df = pd.concat(all_history, ignore_index=True)
pred_test = pd.concat(all_test_predictions, ignore_index=True)
pred_val = pd.concat(all_val_predictions, ignore_index=True)
pred = pd.concat([pred_val, pred_test], ignore_index=True)

MODEL_DIR.mkdir(parents=True, exist_ok=True)
fold_summary.to_csv(MODEL_DIR / 'cmm_folds.csv', index=False)
history_df.to_csv(MODEL_DIR / 'cmm_train_history.csv', index=False)
pred.to_parquet(MODEL_DIR / 'cmm_predictions.parquet', index=False)
if last_model is not None:
    torch.save({
        'state_dict': last_model.state_dict(),
        'feature_cols': feature_cols,
        'return_cols': return_cols,
        'note': 'Last fold model from expanding-window training. Use fold_models/ for each fold.',
    }, MODEL_DIR / 'cmm_model.pt')

print('Saved predictions:', MODEL_DIR / 'cmm_predictions.parquet')
print('Rows:', len(pred), 'Test rows:', len(pred_test), 'Val rows:', len(pred_val))
fold_summary


## 6. Quick Portfolio Check

This is a simple equal-weight decile check on out-of-sample test predictions. A production backtest can later add transaction costs, value weighting, industry neutralization, and tradability filters.


In [ ]:
def decile_backtest(pred_df, signal_col='cmm_signal_cs_z'):
    rows = []
    for date, g in pred_df.groupby('signal_date'):
        if g[signal_col].nunique() < 10:
            continue
        tmp = g.copy()
        tmp['decile'] = pd.qcut(tmp[signal_col].rank(method='first'), 10, labels=False) + 1
        dec = tmp.groupby('decile')['target_1m_ret'].mean()
        rows.append({
            'signal_date': date,
            'long_short': dec.loc[10] - dec.loc[1],
            'long': dec.loc[10],
            'short': dec.loc[1],
        })
    return pd.DataFrame(rows)

bt = decile_backtest(pred_test)
bt.to_csv(MODEL_DIR / 'cmm_test_decile_backtest.csv', index=False)

ann_ret = bt['long_short'].mean() * 12
ann_vol = bt['long_short'].std(ddof=1) * np.sqrt(12)
sharpe = ann_ret / ann_vol if ann_vol > 0 else np.nan
print({'ann_ret': ann_ret, 'ann_vol': ann_vol, 'sharpe': sharpe, 'months': len(bt)})
bt.head()
